# Regularization in Linear Regression: Ridge and Lasso Regression

# 1. Introduction

Regularization is a technique used to prevent overfitting in linear regression by adding a penalty term to the cost function. The penalty discourages the model from assigning excessively large coefficients to the predictor variables, thereby improving its ability to generalize to unseen data.

The two most commonly used regularization techniques are:

- Ridge Regression (L2 Regularization)
- Lasso Regression (L1 Regularization)

Both methods modify the Ordinary Least Squares (OLS) objective function by introducing a regularization parameter that controls the amount of penalty applied.

Regularization introduces a penalty for large regression coefficients.

- Ridge Regression shrinks coefficients towards zero while retaining all predictors.
- Lasso Regression can shrink some coefficients exactly to zero, thereby performing automatic feature selection.


## Mathematical Idea

Ordinary Least Squares minimizes

RSS = Σ(yᵢ − ŷᵢ)²

### Ridge Regression

Objective Function

RSS + α Σβⱼ²

where

- α ≥ 0 is the regularization strength
- βⱼ are the regression coefficients

As α increases, coefficients become progressively smaller but rarely become exactly zero. \
Ridge regression offers a closed form solution and hence, is non-iterative. 


### Lasso Regression

Objective Function

RSS + α Σ|βⱼ|

where

- α ≥ 0 is the regularization strength
- βⱼ are the regression coefficients

As α increases, some coefficients become exactly zero, allowing Lasso to perform feature selection. \
Lasso regression cost function is non-differentiable, and hence, has no closed form solution and is an iterative process.

---

## Algorithm Family

- Supervised
- Parametric
- Eager Learner
- Deterministic
- Batch
- Discriminative


## Ridge vs Lasso

| Ridge Regression | Lasso Regression |
|-----------------|-----------------|
| Uses L2 penalty | Uses L1 penalty |
| Shrinks coefficients | Shrinks and removes coefficients |
| No feature selection | Automatic feature selection |
| Better when all variables contain useful information | Better when only a subset of variables is important |
| More stable with highly correlated features | May arbitrarily select one among correlated features |
| High dimensional datasets | Sparse Datasets |


## Key Takeaways

- Both methods reduce overfitting.
- Ridge keeps every predictor while shrinking coefficients.
- Lasso removes unimportant predictors by assigning zero coefficients.
- The regularization strength (α) controls the trade-off between bias and variance.
- As α increases, model complexity decreases while bias increases.

# 2. Scikit-Learn Implementation

### Dataset
The California Housing Prices dataset contains demographic, geographic and housing-related information collected from California census blocks. The objective is to predict the median house value of a district using its socio-economic and geographical characteristics.

In [24]:
# Import required libraries
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import kagglehub

from sklearn.model_selection import train_test_split, RandomizedSearchCV, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import Ridge, Lasso

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Download dataset
path = kagglehub.dataset_download("camnugent/california-housing-prices")

# Load data
df = pd.read_csv(f"{path}/housing.csv")

display(df.head())

print(df.shape)
print(df.info())

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


(20640, 10)
<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  str    
dtypes: float64(9), str(1)
memory usage: 1.6 MB
None


In [25]:
# Separate features and target
X = df.drop(columns="median_house_value")
y = df["median_house_value"]

# Identify numerical and categorical columns
categorical_features = X.select_dtypes(include="object").columns
numerical_features = X.select_dtypes(exclude="object").columns

# Create preprocessing pipelines
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

# Combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [26]:
def fit_model(model, param_distributions, X_train, y_train):
# Define the model pipeline
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )

    cv = KFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )
# Tune hyperparameters using cross-validation
    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_distributions,
        n_iter=25,
        scoring="neg_root_mean_squared_error",
        cv=cv,
        random_state=42,
        n_jobs=-1
    )
# Train the model
    search.fit(X_train, y_train)

    return search

In [27]:
ridge_params = {
    "model__alpha": np.logspace(-4, 4, 100),
    "model__solver": [
        "auto",
        "svd",
        "cholesky",
        "lsqr",
        "sag"
    ]
}

lasso_params = {
    "model__alpha": np.logspace(-4, 2, 100),
    "model__selection": [
        "cyclic",
        "random"
    ]
}

ridge_search = fit_model(
    Ridge(),
    ridge_params,
    X_train,
    y_train
)

lasso_search = fit_model(
    Lasso(),
    lasso_params,
    X_train,
    y_train
)

print(ridge_search.best_params_)
print(lasso_search.best_params_)

{'model__solver': 'auto', 'model__alpha': np.float64(3.351602650938848)}
{'model__selection': 'random', 'model__alpha': np.float64(9.326033468832199)}


In [28]:
models = {
    "Ridge": ridge_search.best_estimator_,
    "Lasso": lasso_search.best_estimator_
}

results = []

for name, model in models.items():

    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    n = len(y_test)
    p = model.named_steps["preprocessor"].transform(X_train[:1]).shape[1]

    adjusted_r2 = 1 - (
        (1 - r2) * (n - 1)
    ) / (n - p - 1)

    results.append(
        {
            "Model": name,
            "MAE": mae,
            "RMSE": rmse,
            "R²": r2,
            "Adjusted R²": adjusted_r2
        }
    )

results = pd.DataFrame(results)

display(results)

,Model,MAE,RMSE,R²,Adjusted R²
0,Ridge,50683.760770,70076.406021,0.625254,0.624070
1,Lasso,50679.311692,70069.641947,0.625327,0.624143


# Elastic Net Regression

Elastic Net Regression combines the L1 penalty of Lasso Regression and the L2 penalty of Ridge Regression. It performs coefficient shrinkage while simultaneously carrying out feature selection, making it particularly useful when the dataset contains many correlated predictor variables.

Unlike Ridge Regression, Elastic Net can eliminate irrelevant features, and unlike Lasso Regression, it does not arbitrarily select a single feature from a group of highly correlated variables. Instead, correlated features tend to be selected together, often resulting in a more stable model.

### When should Elastic Net be used?

- The dataset contains highly correlated features.
- Feature selection is required.
- Lasso Regression removes too many useful predictors.
- Ridge Regression retains too many irrelevant predictors.

### Important Hyperparameters

| Hyperparameter | Description |
|---------------|-------------|
| alpha | Controls the overall regularization strength. Larger values increase coefficient shrinkage. |
| l1_ratio | Determines the balance between L1 and L2 penalties. A value of 0 corresponds to Ridge Regression, while a value of 1 corresponds to Lasso Regression. |
| max_iter | Maximum number of optimization iterations. |
| tol | Convergence tolerance. |
| fit_intercept | Whether to estimate the intercept term. |

### Interpretation of `l1_ratio`

| l1_ratio | Behaviour |
|-----------|-----------|
| 0 | Equivalent to Ridge Regression |
| 0.25 | Mostly Ridge with slight Lasso behaviour |
| 0.50 | Equal contribution from Ridge and Lasso |
| 0.75 | Mostly Lasso with slight Ridge behaviour |
| 1 | Equivalent to Lasso Regression |

In [29]:
# Import Elastic Net
from sklearn.linear_model import ElasticNet

# Hyperparameter search space
elastic_params = {
    "model__alpha": np.logspace(-4, 2, 100),
    "model__l1_ratio": np.linspace(0.1, 0.9, 9),
    "model__max_iter": [1000, 3000, 5000],
    "model__tol": [1e-3, 1e-4, 1e-5]
}

# Train Elastic Net using the reusable function
elastic_search = fit_model(
    ElasticNet(random_state=42),
    elastic_params,
    X_train,
    y_train
)

# Best hyperparameters
print("Best Parameters:")
print(elastic_search.best_params_)

Best Parameters:
{'model__tol': 1e-05, 'model__max_iter': 3000, 'model__l1_ratio': np.float64(0.6), 'model__alpha': np.float64(0.0004037017258596554)}


In [30]:
# Evaluate Elastic Net
elastic_model = elastic_search.best_estimator_

predictions = elastic_model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

n = len(y_test)
p = elastic_model.named_steps["preprocessor"].transform(X_train.iloc[:1]).shape[1]

adjusted_r2 = 1 - ((1 - r2) * (n - 1)) / (n - p - 1)

print(f"MAE         : {mae:.2f}")
print(f"RMSE        : {rmse:.2f}")
print(f"R² Score    : {r2:.4f}")
print(f"Adjusted R² : {adjusted_r2:.4f}")

MAE         : 50682.55
RMSE        : 70074.11
R² Score    : 0.6253
Adjusted R² : 0.6241


In [31]:
# Compare coefficients from all models
coef_comparison = pd.DataFrame({
    "Feature": ridge_search.best_estimator_.named_steps["preprocessor"].get_feature_names_out(),
    "Ridge": ridge_search.best_estimator_.named_steps["model"].coef_,
    "Lasso": lasso_search.best_estimator_.named_steps["model"].coef_,
    "Elastic Net": elastic_search.best_estimator_.named_steps["model"].coef_
})

coef_comparison = coef_comparison.sort_values(
    by="Ridge",
    key=np.abs,
    ascending=False
)

display(coef_comparison.head(20))

,Feature,Ridge,Lasso,Elastic Net
7,num__median_income,75126.162595,75115.307412,75133.952320
10,cat__ocean_proximity_ISLAND,70190.399993,97649.215111,76473.343527
1,num__latitude,-54222.325589,-54284.570864,-54271.672544
0,num__longitude,-53617.686691,-53667.197253,-53668.385501
9,cat__ocean_proximity_INLAND,-47064.575928,-39868.318581,-48570.341132
5,num__population,-43367.253744,-43340.852277,-43375.116613
4,num__total_bedrooms,42911.028037,42948.042872,42952.360714
6,num__households,18388.264560,18249.749844,18377.569506
2,num__housing_median_age,13902.570865,13886.057437,13900.596073
3,num__total_rooms,-12975.450408,-12900.069902,-12999.304863


# 3. Parameters and Hyperparameters

The implementation of Ridge and Lasso regression in Scikit-learn provides several hyperparameters that control regularization strength, optimization, convergence and model fitting.

---
## Hyperparameters 

### 1. alpha
Default Value: alpha = 1.0 \
Admissible Values: non-negative float \
Controls the strength of regularization
- α = 0 corresponds to Ordinary Least Squares (OLS) regression.
- Larger values impose a stronger penalty on the regression coefficients, resulting in underfitting.
- Moderate values help reduce overfitting
- Always tune α using cross-validation.

### 2. fit_intercept
Default Value: True \
Admissible Values: True/False \
Determines whether an intercept term is estimated.

### 3. max_iter
Default Value: (Ridge: None), (Lasso: 1000) \
Admissible Values: Positive integer \
Maximum number of optimization iterations.
- Small value: Faster training, may fail to converge
- Large value: Longer training, better convergence

### 4. tol
Default Value: 1e-4 \
Admissible Values: Positive float \
Tolerance used to determine convergence.
- Smaller tolerance: More accurate optimization, Longer training time
- Larger tolerance: Faster training,  Slightly less precise solution

### 5. solver (Ridge Only)

Default Value: auto \
Algorithm used to optimize Ridge Regression.

| Solver | Recommended For |
|----------|----------------|
| auto | Automatically selects the best solver |
| svd | Numerically stable for small datasets |
| cholesky | Fast for dense datasets |
| lsqr | Large datasets |
| sag | Very large datasets (requires feature scaling) |
| saga | Large sparse datasets |
| lbfgs | When using positive=True |


### 6. selection (Lasso Only)
Default Value: cyclic
Determines the order in which coefficients are updated during optimization.

| Value | Behaviour |
|--------|-----------|
| cyclic | Updates coefficients sequentially |
| random | Updates coefficients randomly |


### 7. positive
Default Value: False \
Available Values: True/False \
Restricts all regression coefficients to be non-negative.


### Parameter Interactions

- Increasing `alpha` generally requires less model complexity.
- Smaller values of `tol` often require larger values of `max_iter`.
- Very small `alpha` values combined with a low `max_iter` may lead to convergence warnings.


### Key Takeaways

- `alpha` is the most important hyperparameter for both Ridge and Lasso.
- Proper tuning of `alpha` is essential to balance underfitting and overfitting.
- The remaining hyperparameters primarily affect optimization and convergence rather than predictive performance.
- Cross-validation should always be used to identify the optimal regularization strength.

--- 

## Model Parameters

### 1. coef_
Represents the estimated regression coefficients for each predictor variable.

### 2. intercept_
Represents the predicted target value when every feature equals zero.

### 3. n_iter_ (Lasso)
Number of iterations required for the optimization algorithm to converge.
- Small value → Fast convergence.
- Large value → Optimization was more difficult.
- If equal to `max_iter`, the algorithm may not have converged.

### 4. feature_names_in_
Available when training with a Pandas DataFrame. Stores the names of the features used during training.

# 4. Data Requirements

### 1. Feature Scaling
Feature scaling is essential for both Ridge and Lasso Regression. \
The regularization penalty is applied directly to the magnitude of the regression coefficients. If one feature has values ranging from 0–1 while another ranges from 1,000–10,000, the larger-scaled feature will dominate the optimization process and receive a disproportionately large penalty.

### 2. Outliers
Both Ridge and Lasso minimize the Residual Sum of Squares (RSS). Since residuals are squared, observations with large errors have a significant influence on the fitted model. 
- Distorted regression coefficients
- Poor generalization
- Reduced predictive accuracy
- Increased bias in coefficient estimates

### 3. Missing Values
Scikit-learn's Ridge and Lasso implementations cannot train on datasets containing NaN values.

### 4. Categorical Features
The algorithms require numerical input.

### 5. High-Dimensional Data
Both Ridge and Lasso can handle datasets where the number of features is large. \
Lasso is generally preferred when many features are expected to be irrelevant.

### 6. Multicollinearity
One of the strongest use cases for regularization.
- Ridge is usually preferred when correlated predictors all contain useful information.
- Lasso is preferred when feature selection is also desired.
